# 参考文档
- [OpenAI 函数调用文档](https://platform.openai.com/docs/guides/gpt/function-calling)
- [火山引擎 函数调用文档](https://www.volcengine.com/docs/82379/1262342?lang=zh)
- [Kimi 函数调用文档](https://platform.moonshot.cn/docs/guide/use-kimi-api-to-complete-tool-calls#%E4%BB%80%E4%B9%88%E6%98%AF%E5%B7%A5%E5%85%B7%E8%B0%83%E7%94%A8-tool_calls)
- [高德地图 API 文档](https://lbs.amap.com/api/webservice/guide/api/district)

# 相关资料
- [tavily-python](https://github.com/tavily-ai/tavily-python)

# 准备工作
- 申请 Tavily API 密钥
  - 注册账号：[Tavily](https://www.tavily.com/)
  - 获取 API 密钥：[Tavily API 密钥](https://www.tavily.com/docs/api)
- 申请高德地图 API 密钥
  - 注册账号：[高德地图](https://www.amap.com/)
  - 获取 API 密钥：[高德地图 API 密钥](https://www.amap.com/console/show/key)


# 代码开发

In [2]:
# !pip install openai python-dotenv tavily-python requests
from dotenv import load_dotenv
import requests
import os
import json

In [3]:
load_dotenv()

True

## 1. 工具定义

In [4]:
# 1. 定义工具
def get_current_weather(city: str) -> str:
    """获取城市的天气"""
    amap_api_key = os.environ["AMAP_API_KEY"]

    # 获取行政区域编码
    url = f"https://restapi.amap.com/v3/config/district?key={amap_api_key}&keywords={city}"
    response = requests.get(url)
    acode = response.json().get("districts")[0]["adcode"]

    # 获取天气信息
    url = f"https://restapi.amap.com/v3/weather/weatherInfo?key={amap_api_key}&city={acode}&extensions=base"
    response = requests.get(url)
    weather_json = response.json().get("lives")[0]

    return json.dumps(weather_json, ensure_ascii=False, indent=4)

fn_map = {
    'get_current_weather': get_current_weather
}
# 调用工具: get_current_weather("北京")
# 输出：{
#     "province": "北京",
#     "city": "北京市",
#     "adcode": "110000",
#     "weather": "晴",
#     "temperature": "3",
#     "winddirection": "西南",
#     "windpower": "≤3",
#     "humidity": "32",
#     "reporttime": "2026-01-31 18:03:38",
#     "temperature_float": "3.0",
#     "humidity_float": "32.0"
# }

## 2. Schema定义

In [5]:
# 2. 定义工具的schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given city",
            "parameters": {
                "type": "object",
                "required": ["city"],
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "A city name like Beijing or Shanghai"
                    }
                }
            }
        }
    }
]

## 3. 注册工具

In [6]:
# 3. 注册工具
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_BASE_URL"]
    )

resp = client.chat.completions.create(
    model='Qwen/Qwen3-8B',
    messages=[{
        'role': 'user',
        'content': '北京、上海今天天气怎么样'
    }],
    tools=tools,
    tool_choice='auto'
)

## 4. 工具调用

In [ ]:
# 4. 工具调用
tool_call_params = [
    {
        "role": "assistant",
        "tool_calls": [tool_call.model_dump() for tool_call in resp.choices[0].message.tool_calls]
    }
]

print(tool_call_params)

tool_call_result = [
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": fn_map[tool_call.function.name](**json.loads(tool_call.function.arguments))
    }
    for tool_call in resp.choices[0].message.tool_calls
]

print(tool_call_result)

resp_finnal = client.chat.completions.create(
    model='Qwen/Qwen3-8B',
    # 多轮调用，历史记录要带上原始输入 + LLM返回工具调用信息 + 
    messages=[
        {'role': 'user', 'content': '北京、上海今天天气怎么样'}
    ] + tool_call_params + tool_call_result,
    # 开启工具调用
    tools=tools,
    # 自动选择工具
    tool_choice='auto', 
    temperature=0
)
resp_finnal.model_dump()


[{'role': 'assistant', 'tool_calls': [{'id': '019c1c24912e9533c28c88dffcab3efc', 'function': {'arguments': '{"city": "北京"}', 'name': 'get_current_weather'}, 'type': 'function'}, {'id': '019c1c24912e9533c28c88dffcab3efd', 'function': {'arguments': '{"city": "上海"}', 'name': 'get_current_weather'}, 'type': 'function'}]}]
[{'role': 'tool', 'tool_call_id': '019c1c24912e9533c28c88dffcab3efc', 'content': '{\n    "province": "北京",\n    "city": "北京市",\n    "adcode": "110000",\n    "weather": "晴",\n    "temperature": "1",\n    "winddirection": "东",\n    "windpower": "≤3",\n    "humidity": "25",\n    "reporttime": "2026-02-02 10:03:32",\n    "temperature_float": "1.0",\n    "humidity_float": "25.0"\n}'}, {'role': 'tool', 'tool_call_id': '019c1c24912e9533c28c88dffcab3efd', 'content': '{\n    "province": "上海",\n    "city": "上海市",\n    "adcode": "310000",\n    "weather": "晴",\n    "temperature": "6",\n    "winddirection": "北",\n    "windpower": "≤3",\n    "humidity": "63",\n    "reporttime": "2026-0

{'id': '019c1c29c59d3d5106433676074c5dc9',
 'choices': [{'finish_reason': 'stop',
   'index': 0,
   'logprobs': None,
   'message': {'content': '今天北京和上海的天气都是晴天。以下是具体的天气情况：\n\n- **北京**：\n  - 温度：1.0°C\n  - 风向：东\n  - 风力：≤3\n  - 湿度：25.0%\n\n- **上海**：\n  - 温度：6.0°C\n  - 风向：北\n  - 风力：≤3\n  - 湿度：63.0%\n\n两地天气都较为晴朗，但北京的温度较低，建议注意保暖。',
    'refusal': None,
    'role': 'assistant',
    'annotations': None,
    'audio': None,
    'function_call': None,
    'tool_calls': None}}],
 'created': 1769999025,
 'model': 'Qwen/Qwen3-8B',
 'object': 'chat.completion',
 'service_tier': None,
 'system_fingerprint': '',
 'usage': {'completion_tokens': 126,
  'prompt_tokens': 463,
  'total_tokens': 589,
  'completion_tokens_details': {'accepted_prediction_tokens': None,
   'audio_tokens': None,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': None},
  'prompt_tokens_details': None}}

In [97]:
message1 = [
        {'role': 'user', 'content': '北京、上海今天天气怎么样'}
    ] + tool_call_params + tool_call_result,
print(tool_call_params)
print(tool_call_result)
print(json.dumps(message1, indent=4))

[{'role': 'assistant', 'tool_calls': [{'id': '019c1c24912e9533c28c88dffcab3efc', 'function': {'arguments': '{"city": "北京"}', 'name': 'get_current_weather'}, 'type': 'function'}, {'id': '019c1c24912e9533c28c88dffcab3efd', 'function': {'arguments': '{"city": "上海"}', 'name': 'get_current_weather'}, 'type': 'function'}]}]
[{'role': 'tool', 'tool_call_id': '019c1c24912e9533c28c88dffcab3efc', 'content': '{\n    "province": "北京",\n    "city": "北京市",\n    "adcode": "110000",\n    "weather": "晴",\n    "temperature": "1",\n    "winddirection": "东",\n    "windpower": "≤3",\n    "humidity": "25",\n    "reporttime": "2026-02-02 10:03:32",\n    "temperature_float": "1.0",\n    "humidity_float": "25.0"\n}'}, {'role': 'tool', 'tool_call_id': '019c1c24912e9533c28c88dffcab3efd', 'content': '{\n    "province": "上海",\n    "city": "上海市",\n    "adcode": "310000",\n    "weather": "晴",\n    "temperature": "6",\n    "winddirection": "北",\n    "windpower": "≤3",\n    "humidity": "63",\n    "reporttime": "2026-0

In [93]:
# 返回给用户的结果
print(resp_finnal.choices[0].message.content)

今天北京和上海的天气都是晴天。以下是具体的天气情况：

- **北京**：
  - 温度：1.0°C
  - 风向：东
  - 风力：≤3
  - 湿度：25.0%

- **上海**：
  - 温度：6.0°C
  - 风向：北
  - 风力：≤3
  - 湿度：63.0%

两地天气都较为晴朗，但北京的温度较低，建议注意保暖。


# 5. 流式调用

In [32]:
# 4. 工具调用
tool_call_params = [
    {
        "role": "assistant",
        "tool_calls": [tool_call.model_dump() for tool_call in resp.choices[0].message.tool_calls]
    }
]

print(tool_call_params)

tool_call_result = [
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": fn_map[tool_call.function.name](**json.loads(tool_call.function.arguments))
    }
    for tool_call in resp.choices[0].message.tool_calls
]

stream = client.chat.completions.create(
    model='Qwen/Qwen3-8B',
    # 多轮调用，历史记录要带上原始输入 + LLM返回工具调用信息 + 
    messages=[
        {'role': 'user', 'content': '北京、上海今天天气怎么样'}
    ] + tool_call_params + tool_call_result,
    # 开启工具调用
    tools=tools,
    # 自动选择工具
    tool_choice='auto', 
    temperature=0.5,
    stream=True
)


[{'role': 'assistant', 'tool_calls': [{'id': '019c92ae390b1166cea4b3458a3d2ce5', 'function': {'arguments': '{"city": "北京"}', 'name': 'get_current_weather'}, 'type': 'function'}, {'id': '019c92ae390b1166cea4b3458a3d2ce6', 'function': {'arguments': '{"city": "上海"}', 'name': 'get_current_weather'}, 'type': 'function'}]}]


In [33]:
for chunk in stream: 
    print(chunk)

ChatCompletionChunk(id='019c92ce1dec77e7a70b5139b0e3a126', choices=[Choice(delta=ChoiceDelta(content='', function_call=None, refusal=None, role='assistant', tool_calls=None, reasoning_content=None), finish_reason=None, index=0, logprobs=None)], created=1771989507, model='Qwen/Qwen3-8B', object='chat.completion.chunk', service_tier=None, system_fingerprint='', usage=CompletionUsage(completion_tokens=0, prompt_tokens=463, total_tokens=463, completion_tokens_details=None, prompt_tokens_details=None))
ChatCompletionChunk(id='019c92ce1dec77e7a70b5139b0e3a126', choices=[Choice(delta=ChoiceDelta(content='北京', function_call=None, refusal=None, role='assistant', tool_calls=None, reasoning_content=None), finish_reason=None, index=0, logprobs=None)], created=1771989507, model='Qwen/Qwen3-8B', object='chat.completion.chunk', service_tier=None, system_fingerprint='', usage=CompletionUsage(completion_tokens=1, prompt_tokens=463, total_tokens=464, completion_tokens_details=None, prompt_tokens_details

# 小结
我们作为“机器人”的视角对话流程如下：
- 机器人：收到用户请求【询问北京、上海的天气】，并转给模型
- 模型：返回function get_weather({'city':'北京'})、get_weather({'city':'上海'})
- 机器人：调用function get_weather({'city':'北京'})、get_weather({'city':'上海'})，并将结果传给模型
- 模型：返回自然语言，由机器人转给用户